# DiCo-NLI runner (Kaggle)

Đang chạy: **train lại hệ thống chính và lưu model** (`KEEP_MODEL=1`) cho dự đoán test: Qwen EN/ES/EU, Qwen 4 track, mDeBERTa, mỗi cái 3 seed (15 model, không OOF). Sau đó kiểm tra `scripts/predict_final.py` trên dev bằng model đã lưu.

Notebook mỏng: toàn bộ logic nằm trong repo (`scripts/kaggle_queue.sh`).

**Settings:** Accelerator = **GPU T4 x2**, Internet = **On**. Chạy bằng *Save Version → Save & Run All (Commit)* để không phải mở tab.

Hai GPU chạy song song hai hàng đợi (mỗi GPU một process, giữ nguyên batch size của config):
- GPU 0: Qwen 4 track ×3, mDeBERTa seed 1
- GPU 1: Qwen EN/ES/EU ×3, mDeBERTa seed 2, 3

Kết quả:
- Tab *Output*: chỉ tải `runs.zip` (nhỏ, không chứa model), gồm cả `runs/final_check/` (bản kiểm tra trên dev).
- Model (~4–5 GB) nằm trong output của notebook. Tạo Kaggle Dataset từ output này (*Output → New Dataset*, ví dụ `dico-nli-models`) để notebook dự đoán test dùng lại, không cần train lại.


In [ ]:
REPO_URL = 'https://github.com/danh-tc/DiCo-NLI.git'
!git clone --depth 1 $REPO_URL /kaggle/working/dico-nli
%cd /kaggle/working/dico-nli
!git log -1 --oneline

In [ ]:
!pip install -q -e . peft
# peft mới đòi torchao >= 0.16, Kaggle cài sẵn 0.10 -> gỡ torchao (code không dùng nó)
!pip uninstall -y -q torchao
!python -c "import peft; from peft.import_utils import is_torchao_available as f; print('peft', peft.__version__, 'torchao', f())"
!bash scripts/setup_data.sh

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -c "import sys; sys.path.insert(0, 'src'); from dico import env; print(env.describe())"
# Tải model một lần trước, để hai hàng đợi không tải song song
!python -c "from huggingface_hub import snapshot_download as d; d('Qwen/Qwen2.5-1.5B'); d('MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7')"

In [ ]:
%%bash
# Kiểm tra nhanh trước (khoảng 2–3 phút, gồm lưu và nạp lại model): 0.02 epoch Qwen 4 track; lỗi thì dừng, không tốn cả hàng đợi
set -e
CUDA_VISIBLE_DEVICES=0 PYTHONPATH=src python -m dico.train --config configs/qwen_cls_all.yaml --seed 1 --set name=smoke epochs=0.02 "train_tracks=[1]" "eval_tracks=[1]" "decoders=[argmax,self_swap]" keep_model=true > /kaggle/working/smoke.log 2>&1 || { tail -40 /kaggle/working/smoke.log; exit 1; }
# nạp lại model vừa lưu, so với probs lúc train
PYTHONPATH=src python -c "
from pathlib import Path; import numpy as np
from dico.data import load_split; from dico.predict import predict_run
r = Path('/kaggle/working/runs/smoke/seed1'); [(p, s)] = predict_run(r, [load_split('dev', [1])])
ref = np.load(r / 'dev_track1_probs.npy'); print('reload max |dp|', np.abs(p - ref).max(), 'argmax agree', (p.argmax(1) == ref.argmax(1)).mean())
assert (p.argmax(1) == ref.argmax(1)).mean() > 0.99
" > /kaggle/working/smoke_reload.log 2>&1 || { tail -30 /kaggle/working/smoke_reload.log; exit 1; }
grep "reload max" /kaggle/working/smoke_reload.log
rm -rf /kaggle/working/runs/smoke
set +e
echo smoke OK
# Hai hàng đợi song song, mỗi GPU một hàng; xong cả hai mới sang ô sau
# Hệ thống chính, lưu model (15 run, không OOF)
export KEEP_MODEL=1
bash scripts/kaggle_queue.sh 0 train:qwen_cls_all:1 train:qwen_cls_all:2 train:qwen_cls_all:3 train:main:1 &
bash scripts/kaggle_queue.sh 1 train:en:1 train:en:2 train:en:3 train:es:1 train:es:2 train:es:3 train:eu:1 train:eu:2 train:eu:3 train:main:2 train:main:3 &
# Qwen 4 track + OOF (đã chạy 15/11):
# bash scripts/kaggle_queue.sh 0 train:qwen_cls_all:1 train:qwen_cls_all:2 train:qwen_cls_all:3 &
# bash scripts/kaggle_queue.sh 1 oof:qwen_cls_all &
# mDeBERTa (đã chạy 15/11):
# bash scripts/kaggle_queue.sh 0 train:main:1 train:main:2 train:main:3 &
# bash scripts/kaggle_queue.sh 1 oof:main &
# Qwen đơn ngữ (đã chạy 15/11):
# bash scripts/kaggle_queue.sh 0 train:en:1 train:en:2 train:en:3 oof:en train:eu:1 train:eu:2 train:eu:3 &
# bash scripts/kaggle_queue.sh 1 train:es:1 train:es:2 train:es:3 oof:es oof:eu &
wait
cat /kaggle/working/runs/logs/status.txt

In [ ]:
# Nếu có FAIL: xem 40 dòng cuối log của job đó
!grep FAIL /kaggle/working/runs/logs/status.txt || echo "no failures"
!for f in $(grep FAIL /kaggle/working/runs/logs/status.txt | awk '{print $3}'); do echo "== $f"; tail -40 /kaggle/working/runs/logs/$f.log; done

In [ ]:
# Kiểm tra pipeline test trên dev bằng model đã lưu: probs nạp lại phải khớp probs lúc train,
# điểm local phải gần dev_stack_main_v2 (model train lại nên có thể lệch nhẹ)
!PYTHONPATH=src python scripts/predict_final.py --split dev --probs model --runs /kaggle/working/runs --out /kaggle/working/runs/final_check 2>&1 | grep -E "check|rows|local scorer|contains"
!du -sh /kaggle/working/runs/*/seed*/model | sort -h | tail -3; du -sh /kaggle/working/runs

In [ ]:
!cd /kaggle/working && zip -qr runs.zip runs -x '*/ckpt/*' '*/model/*' && ls -lh runs.zip